# Session 3 · Body Mass Index (BMI)

**R for All of Us Bootcamp · Day 2, 10 a.m.–12 p.m. ET**

You will get one BMI per person from **EHR measurements** and **program physical measurements**, and learn to look at units and the distribution before trusting a number.

| Part | Time | What you do |
| --- | --- | --- |
| Demonstration | 10–11 | BMI from EHR: units, range, histogram |
| Lab, Level 1 | 11–11:45 | Program BMI, combine, BMI groups |
| Lab, Level 2 (boss) | if time | Calculate BMI from height and weight |


### How the game works

- Run each **checkpoint** cell after a task. A pass earns points; a miss prints a tip.
- Stuck? Run `hint("ID")` for a clue. Each hint costs 1 point.
- **Level 1** tasks are for everyone. **Level 2 (boss level)** tasks are optional extras for when you finish early.
- Run `scoreboard()` at any time. At the end of the lab, report your **score only** in the Zoom poll.
- Never paste data, counts or screenshots of participant-level data into the Zoom chat.

## R basics refresher

Session 1 has the full introduction. The short version:

- **Run a cell** with **Shift + Enter**. Run cells from top to bottom. `[*]` means the cell is still running.
- `#` starts a **comment**: R ignores the rest of the line. `# TODO` marks where you write code.
- `install.packages("name")` once (buying the textbook); `library(name)` every time you open the notebook (taking it off the shelf).
- **Red text is not always an error.** `Error` stops the cell; `Warning` and messages (such as the tidyverse "Conflicts" list) do not.
- `x <- value` stores a value in an object named `x`. Names are case-sensitive: `AGECAT` is not `agecat`.
- `c("a", "b")` combines values. Text needs quotes; numbers, `TRUE`, `FALSE`, `NA` and object names do not.
- `df$column` picks a column; `df[rows, columns]` picks rows and columns; `df[df$Age_Survey < 25, ]` keeps the rows where the condition is `TRUE`.
- `==` compares (one `=` is only for arguments); `!=` not equal; `&` and; `|` or; `!` not.
- `x %in% c("a", "b")` is `TRUE` when `x` is any of the listed values. View the documentation for `%in%` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/match.html)
- `NA` means missing. Test for it with `is.na(x)`, never `x == NA`. View the documentation for `is.na()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/NA.html)
- `table(x, useNA = "ifany")` counts each value, including `NA`. View the documentation for `table()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/table.html)
- `nrow(df)` counts rows. View the documentation for `nrow()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/nrow.html)

## Setup

Run these cells once each time you open the notebook. If the second cell stops with an error, run `00_SETUP_C2025Q4R6.ipynb` (Python kernel) first, then restart this kernel.

In [ ]:
library(tidyverse)
library(bigrquery)
options(scipen = 999)   # print long concept codes in full, never as 1.2e+15

In [ ]:
# Load the workspace variables written by 00_SETUP_C2025Q4R6.ipynb
env_file <- "/home/jupyter/Setting_Env_Variables_p2.R"
if (!file.exists(env_file)) {
    stop("Run 00_SETUP_C2025Q4R6.ipynb (Python kernel) first, then restart this kernel.")
}
source(env_file)

BILLING <- Sys.getenv("GOOGLE_CLOUD_PROJECT")   # your project: it pays for each query
CDR     <- Sys.getenv("WORKSPACE_CDR")          # the All of Us data, e.g. <project>.C2025Q4R6
BUCKET  <- Sys.getenv("WORKSPACE_BUCKET")       # your workspace bucket (gs://...)

if (any(c(BILLING, CDR, BUCKET) %in% c("", "NOT_FOUND"))) {
    stop("A workspace variable is missing. Re-run 00_SETUP_C2025Q4R6.ipynb, then restart this kernel.")
}

In [ ]:
# ---- Game helpers: run once, no need to read ----
score <- 0
earned <- c()
hints_used <- c()

check <- function(id, ok, points, success, tip) {
    ok <- tryCatch(isTRUE(ok), error = function(e) FALSE)
    if (ok) {
        if (!(id %in% earned)) { score <<- score + points; earned <<- c(earned, id) }
        cat(sprintf("PASS     [%s] %s  (+%d)   Score: %d\n", id, success, points, score))
    } else {
        cat(sprintf("NOT YET  [%s] %s\n", id, tip))
    }
    invisible(ok)
}

hints <- list(
    "3.A" = "Count rows by unit_concept_name. BMI should be in kg/m2; anything else needs a second look.",
    "3.B" = "filter(value_as_number >= bmi_low & value_as_number <= bmi_high), then keep up to 3 most recent dates per person.",
    "3.C" = "Copy the demonstration cleaning cell and change bmidf to probmidf.",
    "3.D" = "bind_rows() stacks the two tables; summarise() then makes one row per person.",
    "3.E" = "BMI groups: < 18.5 Underweight, < 25 Normal, < 30 Overweight, 30 or more Obese.",
    "3.F" = "Keep rows with unit 'centimeter' for height and 'kilogram' for weight, newest date first, slice(1) per person.",
    "3.G" = "BMI = weight in kg / (height in m)^2. Height in m is HTcm / 100."
)

hint <- function(id) {
    if (!(id %in% hints_used)) { score <<- score - 1; hints_used <<- c(hints_used, id) }
    cat(sprintf("HINT [%s] %s\n(-1 point)   Score: %d\n", id, hints[[id]], score))
}

scoreboard <- function() {
    cat(sprintf("Score: %d   Checkpoints passed: %s\n", score,
                if (length(earned)) paste(earned, collapse = ", ") else "none yet"))
}

# Checks used by the checkpoints
one_row_per_person <- function(df) nrow(df) > 0 && !any(duplicated(df$person_id))
only_values <- function(x, allowed) length(x) > 0 && all(na.omit(as.character(x)) %in% allowed)
in_range <- function(x, low, high) any(!is.na(x)) && all(na.omit(x) >= low & na.omit(x) <= high)

In [ ]:
# ---- Save to and load from your workspace bucket ----
save_to_bucket <- function(df, filename) {
    dir.create("data", showWarnings = FALSE)
    local_file <- file.path("data", filename)
    write_excel_csv(df, local_file)
    system(paste0("gsutil cp ", local_file, " ", BUCKET, "/data/"), intern = TRUE)
    message("Saved ", filename, " to ", BUCKET, "/data/")
}

load_from_bucket <- function(filename) {
    dir.create("data", showWarnings = FALSE)
    system(paste0("gsutil cp ", BUCKET, "/data/", filename, " data/"), intern = TRUE)
    read_csv(file.path("data", filename), show_col_types = FALSE)
}

## Study cohort: participants born in 1975 or later

Every query in this notebook keeps only participants **born in 1975 or later**. Here is why we define the cohort by year of birth.

**1. The data have a cutoff date.** CDR v9 (`C2025Q4R6`) "includes participant data with a cutoff date of January 1, 2025"
([CDR v9 release notes](https://support.researchallofus.org/hc/en-us/articles/50653909888788-Curated-Data-Repository-CDR-version-9-Release-Notes)). Data curated after that date are not in this release.
The cutoff is when curation stopped, not a promise that every person has records up to that day.
Cutoff dates for every release are in the [Data Dictionaries](https://support.researchallofus.org/hc/en-us/articles/360033200232-Data-Dictionaries-for-the-Curated-Data-Repositories-CDRs).

**2. Dates of birth are protected.** All of Us changes dates to protect participants' privacy
([Participant Privacy Protections](https://support.researchallofus.org/hc/en-us/articles/4552681983764-Participant-Privacy-Protections); [Data Curation Questions](https://support.researchallofus.org/hc/en-us/articles/35017475171348)):

| Tier | Dates of events (surveys, measurements, diagnoses) | Date of birth |
| --- | --- | --- |
| **Registered** (`R...` datasets) | Shifted backwards by a random 1 to 365 days, the same shift for every date of a participant | Shifted by the same amount |
| **Controlled** (`C...` datasets, used here) | Real dates | **Generalized to year of birth** |

In the Controlled Tier, the year of birth is real, but the month and day of `birth_datetime` are not the participant's.
(Session 1 shows which month and day are stored.)

**3. So we use the year, and only the year.** Born in 1975 or later means a participant was **at most 49 at any point in 2024**,
the last full year before the cutoff. Using one fixed birth year keeps the same people in all four notebooks, whenever the code is run.
A cutoff written as a full date (for example, born after 1975-01-01) would depend on a month and day that are not real.

**4. Ages are computed from years too.** Age at an event = the event's year minus the year of birth. This is the age the person
reaches during that calendar year: it equals their true age, or is one year more if their birthday comes later that year.
It is **never lower** than the true age. The next note explains why that matters.

**Why not subtract dates?** If we subtracted the stored `birth_datetime` from an event date, the answer would depend on the stored month and day.
If that stored day falls later in the year than a participant's real birthday, the computed age comes out too young,
by up to a year. All of Us enrolls adults **18 years or older** ([Join All of Us](https://az.joinallofus.org/en/get-started)),
yet a participant who enrolled shortly after turning 18 could then appear to be **17** at their first survey.
In the Registered Tier the birth date and event dates move by the same shift, so date differences keep the right age,
but the dates themselves no longer match the calendar, so a fixed calendar cutoff would still be off by up to a year.
Year arithmetic avoids both problems: an adult who was 18 at enrollment always comes out as 18 or older.

The filter runs inside BigQuery, so only these participants' rows ever leave Google's servers. That makes every query smaller and faster.

**New in the next cell**

- SQL a query language that BigQuery runs on Google's servers; only the result comes back to your notebook. `SELECT` picks columns, `FROM` names a table, `JOIN ... ON` links two tables on a shared column, `WHERE` keeps the rows that match, and `IN (...)` matches any value in a list or subquery.  
  View the documentation for SQL by clicking [**here.**](https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax)
- `str_glue()` builds text and fills in anything in `{curly braces}` with the value of that R object. We use it to put `CDR`, `FIRST_BIRTH_YEAR` and `UNDER50` into SQL.  
  View the documentation for `str_glue()` by clicking [**here.**](https://stringr.tidyverse.org/reference/str_glue.html)

In [ ]:
CUTOFF <- "2025-01-01"          # CDR v9 data cutoff (release notes)
FIRST_BIRTH_YEAR <- 1975     # born 1975 or later: at most 49 at any point in 2024

# A subquery: the person_id of everyone born in FIRST_BIRTH_YEAR or later.
# year_of_birth is real in the Controlled Tier; the month and day of birth_datetime are not.
# Other queries use it as  WHERE person_id IN ({UNDER50})
UNDER50 <- str_glue("
    SELECT person_id
    FROM `{CDR}.person`
    WHERE year_of_birth >= {FIRST_BIRTH_YEAR}")

In [ ]:
# ---- Run a query once, keep the result in your bucket (run once, no need to read) ----
# First run: BigQuery runs the SQL and saves the result as CSV files in your bucket.
# Later runs: the saved files are read back, with no new query (faster, and no query cost).
# After you change a query, use refresh = TRUE to run it again.
bq_to_df <- function(sql, name, refresh = FALSE) {
    path  <- file.path(BUCKET, "bq_exports", name, paste0(name, "_*.csv"))
    files <- suppressWarnings(system2("gsutil", c("ls", path), stdout = TRUE, stderr = FALSE))
    if (refresh || length(files) == 0 || !is.null(attr(files, "status"))) {
        message("Running the query for ", name, " in BigQuery...")
        bq_table_save(bq_project_query(BILLING, sql), path, destination_format = "CSV")
        files <- system2("gsutil", c("ls", path), stdout = TRUE)
    } else {
        message("Reading saved results for ", name, " (refresh = TRUE re-runs the query)")
    }
    bind_rows(lapply(files, function(f)
        read_csv(pipe(paste("gsutil cat", f)), col_types = cols(.default = col_character()),
                 show_col_types = FALSE)))
}

## Query helpers for this session

These build the SQL for each measurement query. Run the cell once.

**New in the next cell**

- `%>%` (the pipe) passes the result on its left into the function on its right, so steps read top to bottom. Read it as "and then": `df %>% filter(...) %>% mutate(...)`.  
  View the documentation for `%>%` by clicking [**here.**](https://magrittr.tidyverse.org/reference/pipe.html)
- `mutate()` adds a new column or changes an existing one. Every row is kept.  
  View the documentation for `mutate()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/mutate.html)

In [ ]:
# Rewritten from the Data Explorer exports (BMI_V9, BMIfromHTWT_V9) to run faster:
#  - only the 4 columns we use, instead of 26
#  - only the under-50 cohort, and only rows that have a value
#  - concept_ancestor finds child concepts, instead of text searches (LIKE) on the cb_criteria tree
measurement_sql <- function(where_clause) {
    str_glue("
    SELECT m.person_id,
           m.measurement_datetime,
           m.value_as_number,
           u.concept_name AS unit_concept_name
    FROM `{CDR}.measurement` m
    LEFT JOIN `{CDR}.concept` u ON m.unit_concept_id = u.concept_id
    WHERE {where_clause}
      AND m.value_as_number IS NOT NULL
      AND m.person_id IN ({UNDER50})")
}

# Standard concepts and all their children
with_children <- function(concept_ids) {
    ids <- paste(concept_ids, collapse = ", ")
    str_glue("(m.measurement_concept_id IN ({ids})
              OR m.measurement_concept_id IN (SELECT descendant_concept_id FROM `{CDR}.concept_ancestor`
                                              WHERE ancestor_concept_id IN ({ids})))")
}

read_measurements <- function(sql, name) {
    bq_to_df(sql, name) %>%
        mutate(person_id = as.numeric(person_id),
               value_as_number = as.numeric(value_as_number))
}

# Part A · Demonstration

## A1. BMI recorded in the EHR

Standard concepts 3038553, 4245997 and 44783982 (BMI) and their children.

In [ ]:
bmidf <- read_measurements(
    measurement_sql(with_children(c(3038553, 4245997, 44783982))),
    "s3_bmi_ehr_yob1975")

## A2. Query check: units and distribution

Look before you clean. Which units appear, and do the values make sense?

**New in the next cell**

- `group_by()` splits the rows into groups (here, one group per person) so the next step works within each group.  
  View the documentation for `group_by()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/group_by.html)
- `summarise()` / `summarize()` collapses each group to one row, computing values such as `n()` (row count), `mean()` or `max()`.  
  View the documentation for `summarise()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/summarise.html)
- `summary()` a quick look at a numeric column: minimum, quartiles, median, mean, maximum and the number of NAs.  
  View the documentation for `summary()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/summary.html)

In [ ]:
bmidf %>%
    group_by(unit_concept_name) %>%
    summarize(CONCNUM = n(),
              CONCMEAN = mean(value_as_number, na.rm = TRUE),
              CONCMEDIAN = median(value_as_number, na.rm = TRUE))

summary(bmidf$value_as_number)

In [ ]:
check("3.A", exists("bmidf") && "unit_concept_name" %in% names(bmidf),
      5, "You looked at units before cleaning.", "Run the units summary cell above.")

## A3. Outliers: Tukey's fences

Before removing anything, find the values that stand out. **Tukey's method** flags a value as an outlier when it lies more than
**1.5 × IQR** beyond the quartiles:

- **Q1** = the 25th percentile and **Q3** = the 75th percentile; **IQR** = Q3 − Q1, the spread of the middle half of the values
- **lower fence** = Q1 − 1.5 × IQR
- **upper fence** = Q3 + 1.5 × IQR

The fences are measured from the **quartiles, not the mean**. That is what makes the method robust: a few wild errors (a BMI of 250)
barely move the quartiles, but they pull the mean and standard deviation toward themselves, so a "mean ± k × SD" rule can hide the very
errors it is looking for. (The whiskers of a boxplot use the same 1.5 × IQR rule.)

**Fences find values to look at; they don't decide what to remove.** BMI is skewed to the right: the upper fence often lands in the 40s,
and the values above it include real people with severe obesity. So we use Tukey's fences to **screen**, then remove values with a
**plausibility range** written down as a decision (A4).

**New in the next cell**

- `quantile(x, 0.25)` the value below which that share of the data falls: 0.25 gives the 25th percentile (Q1), 0.75 the 75th (Q3). `unname()` drops the "25%" label it attaches.  
  View the documentation for `quantile()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/quantile.html)
- `IQR()` the interquartile range: 75th percentile minus 25th percentile, the spread of the middle half of the values.  
  View the documentation for `IQR()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/IQR.html)
- `ggplot()` starts a plot: the data, plus `aes()` to say which column goes on which axis.  
  View the documentation for `ggplot()` by clicking [**here.**](https://ggplot2.tidyverse.org/reference/ggplot.html)
- `geom_histogram()` draws a histogram: bars showing how many values fall in each bin. `bins` sets how many bars.  
  View the documentation for `geom_histogram()` by clicking [**here.**](https://ggplot2.tidyverse.org/reference/geom_histogram.html)
- `geom_vline()` draws a vertical line on a plot at `xintercept`; `linetype = "dashed"` and `color` change its look.  
  View the documentation for `geom_vline()` by clicking [**here.**](https://ggplot2.tidyverse.org/reference/geom_abline.html)
- `+` in a plot adds a layer or setting to a ggplot. It is not arithmetic here, and it goes at the end of a line, not the start.  
  View the documentation for `+` in a plot by clicking [**here.**](https://ggplot2.tidyverse.org/reference/gg-add.html)
- `theme_classic()` a plain look for the plot: white background, axis lines, no grid lines.  
  View the documentation for `theme_classic()` by clicking [**here.**](https://ggplot2.tidyverse.org/reference/ggtheme.html)

In [ ]:
x <- bmidf$value_as_number

q1  <- unname(quantile(x, 0.25, na.rm = TRUE))   # 25th percentile
q3  <- unname(quantile(x, 0.75, na.rm = TRUE))   # 75th percentile
iqr <- IQR(x, na.rm = TRUE)                      # the same as q3 - q1

lower_fence <- q1 - 1.5 * iqr
upper_fence <- q3 + 1.5 * iqr
round(c(Q1 = q1, Q3 = q3, IQR = iqr, lower_fence = lower_fence, upper_fence = upper_fence), 1)

# Share of records outside the fences (a proportion, not a count)
round(mean(x < lower_fence | x > upper_fence, na.rm = TRUE), 3)

# Histogram (values up to 100) with Tukey's fences (dashed red) and the plausible range we will use, 12 to 60 (solid blue)
ggplot(bmidf[!is.na(x) & x <= 100, ], aes(x = value_as_number)) +
    geom_histogram(bins = 100) +
    geom_vline(xintercept = c(lower_fence, upper_fence), linetype = "dashed", color = "red") +
    geom_vline(xintercept = c(12, 60), color = "blue") +
    theme_classic()

## A4. Clean: plausible range, duplicates, most recent values

**Plausible range: 12 to 60.** Wide enough to keep real underweight and severe obesity; values outside it are almost always entry errors (a weight in pounds, a height in inches, a misplaced decimal). Published rules differ: the VA weight-cleaning algorithm treats BMI of 10 or lower, or 100 or higher, as implausible ([Chan & Raffa 2017, summarized here](https://bookdown.org/revans_evans/weight_algorithms_analyses/03g-ChanRaffa2017.html)), and Guide et al. built a method for erroneous height and weight values in All of Us EHR data ([J Biomed Inform 2024](https://www.researchallofus.org/?p=6460)). Whatever range you choose, write it down and report it in your methods.

**New in the next cell**

- `substr(x, start, stop)` keeps characters `start` to `stop` of a text value: `substr("2023-08-01 09:00:00", 1, 10)` gives `"2023-08-01"`.  
  View the documentation for `substr()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/substr.html)
- `as.Date()` turns text such as `"2023-08-01"` into a date that R can sort, compare and subtract. `format = "%Y-%m-%d"` tells R the layout (year-month-day).  
  View the documentation for `as.Date()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/as.Date.html)
- `filter()` keeps only the rows where the condition is `TRUE`.  
  View the documentation for `filter()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/filter.html)
- `distinct()` removes duplicate rows. With column names, it keeps one row per combination of those columns (`.keep_all = TRUE` keeps the other columns too).  
  View the documentation for `distinct()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/distinct.html)
- `arrange()` sorts the rows. `desc()` sorts largest or newest first.  
  View the documentation for `arrange()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/arrange.html)
- `slice()` keeps rows by position. After `group_by()`, `slice(1)` keeps the first row in each group and `slice(1:3)` keeps up to three.  
  View the documentation for `slice()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/slice.html)
- `ungroup()` removes the grouping, so later steps work on the whole table again. Forgetting it is a common source of surprises.  
  View the documentation for `ungroup()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/group_by.html)

In [ ]:
# Plausible range: written down as a decision, not buried in the code.
# 12 to 60 keeps real underweight and severe (class III) obesity, and removes values that are almost
# always entry errors. State your range in your methods; change it to fit your study.
bmi_low  <- 12
bmi_high <- 60

nrow(bmidf)
bmidf <- bmidf %>%
    mutate(BMIdt = as.Date(substr(measurement_datetime, 1, 10), format = "%Y-%m-%d")) %>%
    filter(value_as_number >= bmi_low & value_as_number <= bmi_high & !is.na(value_as_number)) %>%
    distinct(person_id, BMIdt, value_as_number, .keep_all = TRUE) %>%
    group_by(person_id) %>%
    arrange(desc(BMIdt)) %>%
    slice(1L:3L) %>%                 # up to 3 most recent values per person
    ungroup()
nrow(bmidf)

**New in the next cell**

- `unique()` lists each different value once.  
  View the documentation for `unique()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/unique.html)

In [ ]:
# After cleaning
summary(bmidf$value_as_number)

ggplot(bmidf, aes(x = value_as_number)) +
    geom_histogram(bins = length(unique(bmidf$value_as_number)), alpha = 0.3, color = "orange") +
    theme_classic()

In [ ]:
max_rows <- max(table(bmidf$person_id))
check("3.B", in_range(bmidf$value_as_number, bmi_low, bmi_high) && max_rows <= 3,
      5, "EHR BMI cleaned: in range, at most 3 values per person.", "Values outside the range, or more than 3 rows for someone.")

# Part B · Coding lab

## Level 1

### Get program BMI

Program physical measurements are found by their **source** concept, 903124 (BMI). Run this cell as given.

In [ ]:
probmidf <- read_measurements(
    measurement_sql("m.measurement_source_concept_id = 903124"),
    "s3_bmi_program_yob1975")

probmidf %>%
    group_by(unit_concept_name) %>%
    summarize(CONCNUM = n(), CONCMEDIAN = median(value_as_number, na.rm = TRUE))

### Task 3.C · Clean program BMI

In [ ]:
# TASK 3.C: clean the program BMI values the same way as the EHR values
probmidf <- probmidf %>%
    mutate(BMIdt = as.Date(substr(measurement_datetime, 1, 10), format = "%Y-%m-%d"))
    # TODO: keep values between bmi_low and bmi_high
    # TODO: drop duplicate person/date/value rows
    # TODO: keep up to 3 most recent values per person

summary(probmidf$value_as_number)

In [ ]:
check("3.C", in_range(probmidf$value_as_number, bmi_low, bmi_high) && max(table(probmidf$person_id)) <= 3 &&
             !any(duplicated(probmidf[, c("person_id", "BMIdt", "value_as_number")])),
      15, "Program BMI cleaned.", "Add the filter, distinct and slice steps from the demonstration.")

### Task 3.D · Combine EHR and program BMI

Given code (from BMI_V9). Run it, then run the checkpoint.

In [ ]:
bmialldf <- bind_rows(bmidf, probmidf) %>%
    distinct()

bmialldf1 <- bmialldf %>%
    group_by(person_id) %>%
    arrange(desc(BMIdt)) %>%
    slice(1L:3L) %>%
    summarise(
        BMInumREC = n_distinct(BMIdt),
        BMIdtMIN  = min(BMIdt, na.rm = TRUE),
        BMIdtMAX  = max(BMIdt, na.rm = TRUE),
        BMImn     = mean(value_as_number, na.rm = TRUE),
        BMImed    = median(value_as_number, na.rm = TRUE),
        BMImnTOP3 = mean(head(value_as_number, 3)))

nrow(bmialldf1)

In [ ]:
check("3.D", one_row_per_person(bmialldf1) && in_range(bmialldf1$BMImnTOP3, bmi_low, bmi_high),
      10, "One BMI summary per person.", "person_id repeats in bmialldf1.")

### Task 3.E · BMI groups

**New in Task 3.E**

- `case_when()` a list of `condition ~ value` rules, checked in order; each row gets the value of the first rule that is `TRUE`. `.default` is the value when no rule matches.  
  View the documentation for `case_when()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/case_when.html)

In [ ]:
# TASK 3.E: BMI groups from BMImnTOP3 (mean of up to 3 most recent values)
bmialldf1 <- bmialldf1 %>%
    mutate(BMICAT = case_when(
        # TODO: < 18.5 "Underweight", < 25 "Normal", < 30 "Overweight", 30+ "Obese"
    ))

table(bmialldf1$BMICAT, useNA = "ifany")

In [ ]:
check("3.E", only_values(bmialldf1$BMICAT, c("Underweight", "Normal", "Overweight", "Obese")) &&
             all(c("Normal", "Overweight", "Obese") %in% bmialldf1$BMICAT),
      10, "BMI groups assigned. Level 1 cleared!", "Use the cut points 18.5, 25 and 30.")
save_to_bucket(bmialldf1, "S3_BMI.csv")

## Level 2 (boss level)

### Get program height and weight

Source concepts 903133 (height) and 903121 (weight), from BMIfromHTWT_V9. Run as given.

In [ ]:
htwtdf <- read_measurements(
    measurement_sql("m.measurement_source_concept_id IN (903121, 903133)"),
    "s3_htwt_program_yob1975")

### Task 3.F · Most recent height and weight

**New in Task 3.F**

- `select()` keeps only the columns you name, in the order you name them.  
  View the documentation for `select()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/select.html)

In [ ]:
# BOSS TASK 3.F: most recent height (cm) and weight (kg) per person, side by side
htwtdf1 <- htwtdf %>%
    mutate(HTWTdt = as.Date(substr(measurement_datetime, 1, 10), format = "%Y-%m-%d")) %>%
    distinct(person_id, HTWTdt, value_as_number, .keep_all = TRUE)

table(htwtdf1$unit_concept_name, useNA = "ifany")

dfHT1 <- htwtdf1 %>%
    filter(unit_concept_name == "centimeter")
    # TODO: newest first, one row per person

dfWT1 <- htwtdf1 %>%
    filter(unit_concept_name == "kilogram")
    # TODO: newest first, one row per person

dfHTWT <- full_join(
    dfHT1 %>% select(person_id, HTcm = value_as_number, HTdt = HTWTdt),
    dfWT1 %>% select(person_id, WTkg = value_as_number, WTdt = HTWTdt),
    by = "person_id")

In [ ]:
check("3.F", one_row_per_person(dfHTWT) && all(c("HTcm", "WTkg") %in% names(dfHTWT)),
      10, "Height and weight lined up, one row per person.", "Keep one row per person in dfHT1 and dfWT1 before joining.")

### Task 3.G · Calculate BMI

In [ ]:
# BOSS TASK 3.G: calculate BMI and compare it with the recorded BMI
dfHTWT <- dfHTWT %>%
    mutate(BMIcalc = NA)   # TODO: weight / (height in meters)^2, rounded to 2 decimals

summary(dfHTWT$BMIcalc)

compare <- dfHTWT %>%
    inner_join(bmialldf1 %>% select(person_id, BMImnTOP3), by = "person_id") %>%
    mutate(BMIdiff = BMIcalc - BMImnTOP3)

summary(compare$BMIdiff)
ggplot(compare, aes(x = BMIdiff)) + geom_histogram(bins = 100) + theme_classic()

In [ ]:
check("3.G", all(c("BMIcalc", "BMIdiff") %in% names(compare)) &&
             median(compare$BMIdiff, na.rm = TRUE) > -3 && median(compare$BMIdiff, na.rm = TRUE) < 3,
      20, "Boss defeated: calculated BMI agrees with recorded BMI.", "Check the formula: kg / (cm / 100)^2.")

## Finish line

In [ ]:
scoreboard()
cat("\nReport your score (only your score) in the Zoom poll for Session 3.\n")